# Premier League Match Prediction: Part 1 - Data Preparation & Advanced Feature Engineering

Welcome to the data engineering pipeline for the Premier League Match Predictor.

### Key Improvements Over the Initial Implementation:
1. **Dynamic Club Elo Rating System**: Quantifies team quality and strength-of-schedule (beating Manchester City awards more rating points than beating a newly promoted team). Captures home-pitch advantage and margin-of-victory.
2. **Vectorized Feature Calculations**: Replaces slow $O(N^2)$ manual filtering loops with vectorized `groupby().shift(1).rolling()` transformations (runs in milliseconds with zero future data leakage).
3. **Shot-Quality Signals**: Integrates Shots on Target (SoT), Total Shots, and Shot Ratios. In football, shots on target are significantly more predictive and less prone to sample noise than raw goals.
4. **No Integer Label Encoding of Teams**: Eliminates false ordinal bias (e.g. Arsenal=0, Wolves=24) by characterizing teams through Elo, rolling metrics, and differential features.
5. **Fatigue & Rest Days**: Measures recovery time between fixtures to account for fixture congestion.

### Step 1: Imports & Environment Setup
In the next cell, we import all essential libraries for high-performance tabular data processing.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
print("Libraries successfully imported!")

### Step 2: Multi-Season Data Ingestion & Chronological Sorting
Next, we load the 5 Premier League seasons (`2021/22` through `2025/26`), append season identifiers, concatenate the data, and sort chronologically by date to preserve strict temporal order.

In [ ]:
dataset_dir = Path('dataset')

season_files = [
    ('PL21:22.csv', '2021/22'),
    ('PL22:23.csv', '2022/23'),
    ('PL23:24.csv', '2023/24'),
    ('PL24:25.csv', '2024/25'),
    ('PL25:26.csv', '2025/26')
]

dfs = []
for filename, season in season_files:
    filepath = dataset_dir / filename
    if filepath.exists():
        season_df = pd.read_csv(filepath)
        season_df['Season'] = season
        dfs.append(season_df)
        print(f"Loaded {filename}: {len(season_df)} matches")
    else:
        print(f"Warning: {filename} not found at {filepath}")

df_raw = pd.concat(dfs, ignore_index=True)
df_raw['Date'] = pd.to_datetime(df_raw['Date'])
df_raw = df_raw.sort_values(by='Date').reset_index(drop=True)
df_raw['Match_ID'] = df_raw.index

print(f"\nTotal matches loaded: {len(df_raw)}")
df_raw[['Season', 'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR']].head()

### Step 3: Dynamic Club Elo Rating System
The standard Elo rating model updates dynamically after each match:

$$E_{\text{home}} = \frac{1}{1 + 10^{-(R_{\text{home}} + H_{\text{adv}} - R_{\text{away}})/400}}$$

$$\Delta R = K \times G \times (S - E)$$

where:
- $H_{\text{adv}} = 65$: Home pitch advantage rating bonus.
- $K = 24$: Sensitivity parameter.
- $G$: Margin of victory multiplier ($G = 1 + 0.5 \times (|\text{GoalDiff}| - 1)$) to reward convincing wins.
- $S$: Match score ($1.0$ for win, $0.5$ for draw, $0.0$ for loss).

**Strict No-Leakage Rule**: The ratings assigned to a match are strictly the *pre-match* ratings before the result is known.

In [ ]:
class FootballElo:
    def __init__(self, base_rating=1500.0, k_factor=24.0, home_advantage=65.0):
        self.base_rating = base_rating
        self.k_factor = k_factor
        self.home_advantage = home_advantage
        self.ratings = {}

    def get_rating(self, team):
        return self.ratings.get(team, self.base_rating)

    def expected_score(self, r_home, r_away):
        dr = (r_home + self.home_advantage) - r_away
        e_home = 1.0 / (1.0 + 10.0 ** (-dr / 400.0))
        return e_home, 1.0 - e_home

    def update(self, home_team, away_team, fthg, ftag):
        r_home = self.get_rating(home_team)
        r_away = self.get_rating(away_team)
        
        e_home, e_away = self.expected_score(r_home, r_away)
        
        if fthg > ftag:
            s_home, s_away = 1.0, 0.0
        elif fthg < ftag:
            s_home, s_away = 0.0, 1.0
        else:
            s_home, s_away = 0.5, 0.5
            
        goal_diff = abs(fthg - ftag)
        # Margin of victory multiplier
        if goal_diff <= 1:
            margin_mult = 1.0
        elif goal_diff == 2:
            margin_mult = 1.5
        else:
            margin_mult = 1.75 + (goal_diff - 3) / 8.0
            
        delta_home = self.k_factor * margin_mult * (s_home - e_home)
        delta_away = self.k_factor * margin_mult * (s_away - e_away)
        
        self.ratings[home_team] = r_home + delta_home
        self.ratings[away_team] = r_away + delta_away

# Compute pre-match Elo for each game
elo_system = FootballElo()
home_elos = []
away_elos = []

for _, row in df_raw.iterrows():
    h_team, a_team = row['HomeTeam'], row['AwayTeam']
    # Pre-match snapshot
    home_elos.append(elo_system.get_rating(h_team))
    away_elos.append(elo_system.get_rating(a_team))
    # Post-match update
    elo_system.update(h_team, a_team, row['FTHG'], row['FTAG'])

df_raw['Home_Elo'] = home_elos
df_raw['Away_Elo'] = away_elos
df_raw['Elo_Diff'] = df_raw['Home_Elo'] - df_raw['Away_Elo']

print("Sample of Elo tracking:")
df_raw[['Date', 'HomeTeam', 'AwayTeam', 'Home_Elo', 'Away_Elo', 'Elo_Diff', 'FTR']].head()

### Step 4: Vectorized Team-Match Reshaping
To compute rolling statistics without slow nested loops, we reshape the dataset into a **team-game perspective table** (2 rows per match: one for the home team, one for the away team).
This allows all calculations to be performed cleanly with vectorized Pandas `groupby` operations.

In [ ]:
# Home team perspective
home_recs = df_raw[['Match_ID', 'Date', 'Season', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC']].copy()
home_recs.columns = ['Match_ID', 'Date', 'Season', 'Team', 'Opponent', 'GF', 'GA', 'SF', 'SA', 'STF', 'STA', 'CF', 'CA']
home_recs['Is_Home'] = 1
home_recs['Points'] = np.where(home_recs['GF'] > home_recs['GA'], 3, np.where(home_recs['GF'] == home_recs['GA'], 1, 0))

# Away team perspective
away_recs = df_raw[['Match_ID', 'Date', 'Season', 'AwayTeam', 'HomeTeam', 'FTAG', 'FTHG', 'AS', 'HS', 'AST', 'HST', 'AC', 'HC']].copy()
away_recs.columns = ['Match_ID', 'Date', 'Season', 'Team', 'Opponent', 'GF', 'GA', 'SF', 'SA', 'STF', 'STA', 'CF', 'CA']
away_recs['Is_Home'] = 0
away_recs['Points'] = np.where(away_recs['GF'] > away_recs['GA'], 3, np.where(away_recs['GF'] == away_recs['GA'], 1, 0))

# Combine and sort chronologically per team
team_matches = pd.concat([home_recs, away_recs], ignore_index=True)
team_matches = team_matches.sort_values(by=['Team', 'Date']).reset_index(drop=True)
print(f"Team perspective records created: {len(team_matches)}")
team_matches.head()

### Step 5: Advanced Rolling Features (Form, Goals, Shots, Rest Days)
We now calculate:
1. **Strict Prior-Game Shift (`.shift(1)`)**: Prevents any current-game leakage.
2. **Last 5 Games Form Points** (`Form_5`).
3. **Rolling Averages (last 5 games)**:
   - Goals For (`GF_5`) & Goals Against (`GA_5`).
   - Goal Difference (`GD_5`).
   - Shots on Target For (`STF_5`) & Against (`STA_5`).
   - Shots on Target Ratio: $\frac{\text{STF}}{\text{STF} + \text{STA}}$ (shot dominance).
4. **Rest Days**: Number of days between consecutive matches (fatigue proxy).
5. **Venue-Specific Form**: Performance strictly when playing Home (for home games) or Away (for away games).

In [ ]:
# 1. Days of rest since previous match
team_matches['Prev_Date'] = team_matches.groupby('Team')['Date'].shift(1)
team_matches['Rest_Days'] = (team_matches['Date'] - team_matches['Prev_Date']).dt.days
team_matches['Rest_Days'] = team_matches['Rest_Days'].fillna(7).clip(upper=21)

# 2. Overall Rolling Stats (Last 5 matches, shifted by 1 match)
grouped = team_matches.groupby('Team')

team_matches['Roll_Points_5'] = grouped['Points'].shift(1).rolling(5, min_periods=1).sum()
team_matches['Roll_GF_5'] = grouped['GF'].shift(1).rolling(5, min_periods=1).mean()
team_matches['Roll_GA_5'] = grouped['GA'].shift(1).rolling(5, min_periods=1).mean()
team_matches['Roll_GD_5'] = team_matches['Roll_GF_5'] - team_matches['Roll_GA_5']

team_matches['Roll_STF_5'] = grouped['STF'].shift(1).rolling(5, min_periods=1).mean()
team_matches['Roll_STA_5'] = grouped['STA'].shift(1).rolling(5, min_periods=1).mean()

# Shot on Target Dominance Ratio (safe division)
total_st = team_matches['Roll_STF_5'] + team_matches['Roll_STA_5']
team_matches['Roll_STRatio_5'] = np.where(total_st > 0, team_matches['Roll_STF_5'] / total_st, 0.5)

# 3. Venue-Specific Rolling Points (Home form for Home matches, Away form for Away matches)
team_matches['Venue_Points_5'] = team_matches.groupby(['Team', 'Is_Home'])['Points'].shift(1).rolling(5, min_periods=1).sum()

# Fill initial matches with league baseline averages
team_matches['Roll_Points_5'] = team_matches['Roll_Points_5'].fillna(6.0)
team_matches['Roll_GF_5'] = team_matches['Roll_GF_5'].fillna(1.3)
team_matches['Roll_GA_5'] = team_matches['Roll_GA_5'].fillna(1.3)
team_matches['Roll_GD_5'] = team_matches['Roll_GD_5'].fillna(0.0)
team_matches['Roll_STF_5'] = team_matches['Roll_STF_5'].fillna(4.0)
team_matches['Roll_STA_5'] = team_matches['Roll_STA_5'].fillna(4.0)
team_matches['Roll_STRatio_5'] = team_matches['Roll_STRatio_5'].fillna(0.5)
team_matches['Venue_Points_5'] = team_matches['Venue_Points_5'].fillna(6.0)

print("Vectorized rolling features calculated successfully!")
team_matches[['Team', 'Date', 'Is_Home', 'Rest_Days', 'Roll_Points_5', 'Roll_GF_5', 'Roll_GA_5', 'Roll_STF_5', 'Roll_STRatio_5']].tail()

### Step 6: Merging Features Back to Match Level & Engineering Differentials
Now we separate the home perspective and away perspective and merge them back onto `Match_ID`.
We also construct **Differential Features**, which provide clean relative advantages that linear models and tree-based models leverage with maximum signal-to-noise ratio.

In [ ]:
feature_cols = [
    'Match_ID', 'Team', 'Rest_Days', 'Roll_Points_5', 'Roll_GF_5', 'Roll_GA_5',
    'Roll_GD_5', 'Roll_STF_5', 'Roll_STA_5', 'Roll_STRatio_5', 'Venue_Points_5'
]

home_features = team_matches[team_matches['Is_Home'] == 1][feature_cols].copy()
home_features.columns = ['Match_ID', 'HomeTeam'] + [f'Home_{c}' for c in feature_cols[2:]]

away_features = team_matches[team_matches['Is_Home'] == 0][feature_cols].copy()
away_features.columns = ['Match_ID', 'AwayTeam'] + [f'Away_{c}' for c in feature_cols[2:]]

# Merge with raw match dataframe
dataset = df_raw.merge(home_features, on=['Match_ID', 'HomeTeam'], how='left')
dataset = dataset.merge(away_features, on=['Match_ID', 'AwayTeam'], how='left')

# Differential features
dataset['Diff_Form'] = dataset['Home_Roll_Points_5'] - dataset['Away_Roll_Points_5']
dataset['Diff_Goals'] = dataset['Home_Roll_GF_5'] - dataset['Away_Roll_GA_5']
dataset['Diff_GD'] = dataset['Home_Roll_GD_5'] - dataset['Away_Roll_GD_5']
dataset['Diff_ST'] = dataset['Home_Roll_STF_5'] - dataset['Away_Roll_STA_5']
dataset['Diff_STRatio'] = dataset['Home_Roll_STRatio_5'] - dataset['Away_Roll_STRatio_5']
dataset['Diff_Venue_Strength'] = dataset['Home_Venue_Points_5'] - dataset['Away_Venue_Points_5']
dataset['Diff_Rest_Days'] = dataset['Home_Rest_Days'] - dataset['Away_Rest_Days']

# Map target labels: H:0, D:1, A:2
dataset['Target'] = dataset['FTR'].map({'H': 0, 'D': 1, 'A': 2})

print(f"Final dataset shape: {dataset.shape}")
dataset[['Date', 'HomeTeam', 'AwayTeam', 'Elo_Diff', 'Diff_Form', 'Diff_ST', 'Diff_STRatio', 'FTR', 'Target']].head()

### Step 7: Data Validation & Export
Before saving, we verify there are no missing values in our final feature matrix and export the processed dataset to `dataset/processed_match_dataset.csv`.

In [ ]:
output_file = Path('dataset/processed_match_dataset.csv')
dataset.to_csv(output_file, index=False)
print(f"Engineered dataset saved to {output_file} ({os.path.getsize(output_file) / 1024:.1f} KB)")

# Outcome Distribution
outcome_counts = dataset['FTR'].value_counts(normalize=True) * 100
print("\nHistorical Match Outcome Distribution:")
print(f"Home Win (H): {outcome_counts.get('H', 0):.2f}%")
print(f"Draw (D):     {outcome_counts.get('D', 0):.2f}%")
print(f"Away Win (A): {outcome_counts.get('A', 0):.2f}%")